In [ ]:
import os, itertools
import numpy as np, pandas as pd, torch
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             precision_recall_fscore_support)
)

# ------------------------------ CONFIG -------------------------------
DIM   = 32
BASE  = '/'
IN_PT = f'{BASE}/gathered_{DIM}/all_users_gathered{DIM}.pt'

RATIOS = (0.7, 0.2, 0.1)
TRAIN_NORMAL_ONLY = True
SEED = 42

GRID = {'n_estimators': [300],
        'max_samples':  [256, 1024],
        'max_features': [0.3, 0.6, 1.0]}

ERRORS = []
def err(m): ERRORS.append(m); print('[ERROR]', m)
def ok(m):  print('[OK   ]', m)
def halt(c):
    if ERRORS: raise RuntimeError(f'{c}: ERROR {len(ERRORS)}건\n'
                                  + '\n'.join('  - ' + e for e in ERRORS))

# --------------------------- 1. 로딩 / 검증 ---------------------------
if not os.path.isfile(IN_PT): raise FileNotFoundError(f'입력 없음: {IN_PT}')
if abs(sum(RATIOS) - 1.0) > 1e-9: raise ValueError(f'RATIOS 합이 1이 아니다: {RATIOS}')

d = torch.load(IN_PT, map_location='cpu', weights_only=False)
X     = d['X'].numpy().astype(np.float64)
MASK  = d['mask'].numpy()
y     = d['label'].numpy().astype(int)
users = np.array(d['users'])
names = list(d['feature_names']); offsets = d['user_offsets']
NSTAT, MODS = int(d['n_stat']), list(d['modalities'])
N, D = X.shape

if int(d['n_components']) != DIM: err(f'파일 DIM={d["n_components"]} != CONFIG DIM={DIM}')
if len(names) != D:               err(f'feature_names {len(names)} != D {D}')
if MASK.shape != X.shape:         err('mask shape 불일치')
if X[~MASK].any():                err('mask=False 위치에 0이 아닌 값')
if not np.isfinite(X).all():      err('X 에 NaN/Inf')
halt('입력 검증')
ok(f'X {X.shape} | users {len(set(users))} | positives {y.sum():,} ({y.mean()*100:.2f}%)')

# --------------------------- 2. feature set ---------------------------
STAT_IDX = list(range(NSTAT))
FLAG_IDX = [names.index(f'{m}_missing') for m in MODS]
EMB_IDX  = [i for i in range(D) if i not in set(STAT_IDX + FLAG_IDX)]
FEATURE_SETS = {'stat':      STAT_IDX,
                'stat+flag': STAT_IDX + FLAG_IDX,
                'emb+flag':  EMB_IDX + FLAG_IDX,
                'all':       list(range(D))}

# ------------------------ 3. 사용자별 시간 3분할 -----------------------
tr, va, te = [], [], []
for u in d['user_order']:
    s, e = offsets[u]; n = e - s
    c1 = s + int(round(n * RATIOS[0]))
    c2 = s + int(round(n * (RATIOS[0] + RATIOS[1])))
    tr += list(range(s, c1)); va += list(range(c1, c2)); te += list(range(c2, e))
tr, va, te = map(np.array, (tr, va, te))
tr_fit = tr[y[tr] == 0] if TRAIN_NORMAL_ONLY else tr

print(f'\nsplit = {RATIOS[0]:.0%}/{RATIOS[1]:.0%}/{RATIOS[2]:.0%} (사용자별 시간순)')
for nm, ix in [('train', tr), ('val', va), ('test', te)]:
    print(f'  {nm:5s} {len(ix):5,d} rows  positives {int(y[ix].sum()):4,d} '
          f'({y[ix].mean()*100:5.2f}%)  users {len(set(users[ix]))}')
print(f'  fit 사용 {len(tr_fit):,} rows (train positive {len(tr)-len(tr_fit):,}건 제외)')

if len(set(tr) & set(va)) or len(set(va) & set(te)) or len(set(tr) & set(te)):
    err('split 인덱스가 겹친다')
if len(tr) + len(va) + len(te) != N: err('split 합이 전체 row 수와 다르다')
if y[va].sum() == 0: err('val 에 positive 가 없다')
if y[te].sum() == 0: err('test 에 positive 가 없다')
if y[te].sum() < 20: print(f'[WARN ] test positive {int(y[te].sum())}건뿐이라 지표 분산이 크다')
halt('split 검증')

# ---------------------------- 4. 평가 함수 ----------------------------
def pick_threshold(y_true, score):
    """val 에서 F1 을 최대화하는 임계값."""
    prec, rec, thr = precision_recall_curve(y_true, score)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return float(thr[int(np.nanargmax(f1[:-1]))])

def evaluate(y_true, score, thr):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, (score >= thr).astype(int), average='binary', zero_division=0)
    return {'precision': p, 'recall': r, 'F1': f,
            'PR_AUC': average_precision_score(y_true, score)}

# ------------------------ 5. val 기반 그리드 탐색 ----------------------
combos = [dict(zip(GRID, v)) for v in itertools.product(*GRID.values())]
print(f'\n탐색: feature set {len(FEATURE_SETS)}개 x 파라미터 {len(combos)}개 '
      f'= {len(FEATURE_SETS)*len(combos)} fit')

search, cache = [], {}
for tag, cols in FEATURE_SETS.items():
    for pr in combos:
        clf = IsolationForest(**pr, contamination='auto',
                              random_state=SEED, n_jobs=-1).fit(X[np.ix_(tr_fit, cols)])
        s = -clf.score_samples(X[:, cols])                 # 클수록 이상
        search.append({'feature_set': tag, 'n_features': len(cols), **pr,
                       'val_PR_AUC': average_precision_score(y[va], s[va])})
        cache[(tag, tuple(pr.items()))] = s

print('\n--- validation PR-AUC (상위 10) ---')
print(pd.DataFrame(search).sort_values('val_PR_AUC', ascending=False)
      .set_index(['feature_set', 'max_samples', 'max_features'])[['n_features', 'val_PR_AUC']]
      .head(10).round(4).to_string())

best = max(search, key=lambda r: r['val_PR_AUC'])
print(f'\n선택: feature_set={best["feature_set"]} '
      f'max_samples={best["max_samples"]} max_features={best["max_features"]} '
      f'-> val PR-AUC {best["val_PR_AUC"]:.4f}')

# --------------------- 6. test 최종 평가 (1회) -------------------------
rows = []
def add(name, score, nfeat):
    thr = pick_threshold(y[va], score[va])          # 임계값은 val 에서만 결정
    rows.append({'model': name, 'n_features': nfeat, **evaluate(y[te], score[te], thr)})

for tag in FEATURE_SETS:
    b = max((r for r in search if r['feature_set'] == tag), key=lambda r: r['val_PR_AUC'])
    mark = ' *' if tag == best['feature_set'] else ''
    add(f'IF[{tag}]{mark}', cache[(tag, tuple((k, b[k]) for k in GRID))], b['n_features'])
add('trivial[file활동]', 1.0 - X[:, names.index('file_missing')], 1)


res = (pd.DataFrame(rows).set_index('model')
       [['n_features', 'precision', 'recall', 'F1', 'PR_AUC']]
       .sort_values('PR_AUC', ascending=False).round(4))

print('\n' + '=' * 62)
print(f'TEST  (rows {len(te):,} / positives {int(y[te].sum()):,} '
      f'/ positive 비율 {y[te].mean()*100:.2f}%)')
print('=' * 62)
print(res.to_string())
print('\n* = val PR-AUC 기준 선택된 조합. 임계값은 val 의 F1 최대점에서 결정 후 test 에 적용.')